# Chartrand et al. (2023): human against mouse L1 cells

**Paper.** Chartrand et al. (2023), *Science* 382, eadf0805,
[doi:10.1126/science.adf0805](https://doi.org/10.1126/science.adf0805); see
`reproduce_chartrand_2023.ipynb` for the text read, the human data and the table of all claims.

**Data.** Human cells from [DANDI:000630](https://dandiarchive.org/dandiset/000630/0.230915.2257)
version 0.230915.2257, the dataset under test. Mouse cells from
[DANDI:000020](https://dandiarchive.org/dandiset/000020/0.210913.1639) version 0.210913.1639, the
outside input the claims file names (with that version) for the mouse comparison; it is the only
published version of 000020. Mouse labels (t-type, layer, structure) and the mouse features the
files do not give come from the paper's code at commit 7777857c1b89b19735e45fd9c9bd7493cc1d805c.
Only the files of the paper's 272 strict mouse L1 cells are read.

The claims here use outside data (`data: with_inputs`) and are counted apart from the claims on
the dandiset alone. Features follow the claims file's O1 as in the human notebook: (a) computed
here where they agree with the paper's tables for both species, the paper's values otherwise; (b)
the paper's values.

| Claim | Status |
|---|---|
| mouse-cell-link (added link claim) | tested |
| fig3d-ephys-species | tested |
| fig3g-variability-dimensions | tested |
| fig4d-mouse-no-rosehip-ephys | tested |
| fig5d-mouse-pax6-doublets | tested |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

import l1ephys as L

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")
started = time.time()

# Human cells (dandiset 000630), as in reproduce_chartrand_2023.ipynb
h_assets = L.list_assets()
h_feats = L.all_cell_features(h_assets, workers=12)
labels = L.load_labels()
paper_h = L.load_paper_features()
print(f"human: dandiset {L.DANDISET_ID} version {L.DANDISET_VERSION}, {len(h_feats)} files read "
      f"in {time.time() - started:.0f} s")

# Mouse cells (dandiset 000020), only the files of the paper's mouse L1 cells
mlabels = L.load_mouse_labels()
paper_m = L.load_mouse_paper_features()
m_meta = L.version_metadata(L.MOUSE_DANDISET_ID, L.MOUSE_DANDISET_VERSION)
m_assets_all = L.list_assets(L.MOUSE_DANDISET_ID, L.MOUSE_DANDISET_VERSION)
m_assets_all["session_id"] = pd.to_numeric(m_assets_all.path.str.extract(r"ses-(\d+)")[0])
wanted = mlabels[mlabels.in_strict_set].ephys_roi_id
m_assets = m_assets_all[m_assets_all.session_id.isin(wanted)]
print(f"mouse: dandiset {L.MOUSE_DANDISET_ID} version {L.MOUSE_DANDISET_VERSION} ({m_meta['name']!r}), "
      f"{len(m_assets_all)} assets; files of the {len(wanted)} strict mouse L1 cells found: {len(m_assets)}")
t1 = time.time()
m_feats = L.all_cell_features(m_assets, workers=12)
print(f"mouse features from {len(m_feats)} files in {time.time() - t1:.0f} s")

## Data check

In [ ]:
print("Mouse files read:", len(m_feats), "| errors:", int(m_feats.get("error", pd.Series(dtype=str)).notna().sum()))
print("Species:", m_feats.species.value_counts().to_dict(), "| institution:", m_feats.institution.value_counts().to_dict())
print("Subjects:", m_feats.subject_id.nunique(), "| sessions:", m_feats.session_id.nunique())
print("Sweeps used per file:", m_feats.n_sweeps_used.describe()[["min", "50%", "max"]].to_dict(),
      "| zero-padded sweeps left out:", int(m_feats.n_sweeps_zero_padded.sum()))
print("Stimulus onset (s):", m_feats.stim_start_s.value_counts().to_dict(), "| duration (s):",
      m_feats.stim_dur_s.round(3).value_counts().to_dict())
print("Clock check, median voltage change at the true onset and 0.3 s early (mV):",
      round(m_feats.clock_dv_true_mv.median(), 2), round(m_feats.clock_dv_shifted_mv.median(), 2))
print("Strict mouse L1 cells in the paper's table:", int(mlabels.in_strict_set.sum()),
      "| by structure:", mlabels[mlabels.in_strict_set].structure.str.extract(r"^([A-Za-z]+)")[0].value_counts().to_dict())

## Mouse cells against the paper's mouse table (link claim `mouse-cell-link`)

> The files of DANDI 000020 can be matched one to one to the paper's mouse L1 cells
> (mouse_l1_dataset_2023_02_06.csv), and the matched records agree on what both describe.

In [ ]:
m_feats["session_id_int"] = m_feats.session_id.astype("int64")
by_roi = mlabels[mlabels.in_strict_set].reset_index().dropna(subset=["ephys_roi_id"]).astype({"ephys_roi_id": "int64"}).set_index("ephys_roi_id")
mlinked = m_feats.join(by_roi, on="session_id_int", rsuffix="_table")
print("Mouse files matched by session id:", int(mlinked.spec_id.notna().sum()), "of", len(mlinked),
      "| distinct cells:", mlinked.spec_id.nunique())
print("Cell name in the file equal to the table's cell_name:", int((mlinked.cell_name == mlinked.cell_name_table).sum()))
missing = mlabels[mlabels.in_strict_set & ~mlabels.ephys_roi_id.isin(m_feats.session_id_int)]
print("Strict mouse cells without a file in 000020:", len(missing), missing.structure.value_counts().to_dict())
mcomputed = L.with_adapt_order(mlinked.set_index("spec_id"))
fcols = L.feature_columns(paper_m)
rows = []
for c in [c for c in fcols if c in mcomputed.columns]:
    a, b = mcomputed[c].astype(float), paper_m[c].reindex(mcomputed.index).astype(float)
    ok = a.notna() & b.notna()
    rows.append({"feature": c, "n_both": int(ok.sum()),
                 "within_1pct": round((np.abs(a[ok] - b[ok]) <= 0.01 * np.abs(b[ok]) + 1e-9).mean(), 3)})
mcmp = pd.DataFrame(rows).set_index("feature")
print(mcmp.sort_values("within_1pct").to_string())

In [ ]:
# The human tables, built as in reproduce_chartrand_2023.ipynb
h_feats["session_id_int"] = h_feats.session_id.astype("int64")
hl = h_feats.join(labels.reset_index().dropna(subset=["ephys_roi_id"]).astype({"ephys_roi_id": "int64"}).set_index("ephys_roi_id"), on="session_id_int", rsuffix="_table")
hcomputed = L.with_adapt_order(hl.set_index("spec_id"))
def agree(comp, pap):
    out = []
    for c in [c for c in fcols if c in comp.columns]:
        a, b = comp[c].astype(float), pap[c].reindex(comp.index).astype(float)
        ok = a.notna() & b.notna()
        if ok.sum() and (np.abs(a[ok] - b[ok]) <= 0.01 * np.abs(b[ok]) + 1e-9).mean() >= 0.5:
            out.append(c)
    return out
H_COMP, M_COMP = agree(hcomputed, paper_h), agree(mcomputed, paper_m)
COMPUTED = sorted(set(H_COMP) & set(M_COMP))
print("Features computed here and used for both species:", len(COMPUTED), "| the others from the paper's tables")

def build(kind):
    hf = L.apply_code_filters(L.hybrid_table(hcomputed, paper_h, COMPUTED) if kind == "a" else paper_h)
    mf = L.apply_code_filters(L.hybrid_table(mcomputed, paper_m, COMPUTED) if kind == "a" else paper_m)
    H = hl.set_index("spec_id").drop(columns=[c for c in fcols if c in hl.columns]).join(hf[fcols])
    H["homology_type"] = H["t-type"].map(L.HOMOLOGY)
    H["failed_fx_long_squares"] = paper_h["failed_fx_long_squares"].reindex(H.index)
    H["layer"] = H.layer.fillna(H.layer_lims.astype("string").str.replace(".0", "", regex=False))
    H["species"] = "human"
    M = mlinked.set_index("spec_id").drop(columns=[c for c in fcols if c in mlinked.columns]).join(mf[fcols])
    M["failed_qc"] = paper_m["failed_qc"].reindex(M.index)
    M["vis"] = M.structure.str.contains("VIS")
    M["species"] = "mouse"
    return H, M
HA, MA = build("a")
HB, MB = build("b")
print("Human cells:", len(HA), "| mouse cells:", len(MA), "| mouse VIS with failed_qc False:",
      int((MA.vis & (MA.failed_qc == False)).sum()))
print("Mouse cells by subclass (VIS, QC passed):", MA[MA.vis & (MA.failed_qc == False)].homology_type.value_counts().to_dict())
print("Mouse cells by subclass and layer_ccf:"); print(pd.crosstab(MA.homology_type.fillna("none"), MA.layer_ccf.fillna("NA")).to_string())

**Result: partially reproduced.** All 250 files of 000020 that belong to the paper's strict
mouse L1 cells match one cell each, by session id and by cell name, and their features agree with
the paper's mouse table within 1% for 88 to 100% of cells for most features (the adaptation
ratios and f-I slope agree less). 22 of the 272 strict mouse cells, 20 of them in VISp layer 1,
have no file in this version of 000020, so the mouse claims below use 250 cells, 235 of them
passing QC.

## fig3d-ephys-species

> Mouse L1 cells have less sag, lower spike amplitude and higher rheobase than human, across
> subclasses and within at least 2 of the 4 matched subclasses.

As l1_cross_species.ipynb cells 5, 25 and 38: human AIBS cells with long square features and
soma in L1 (layer, filled from the LIMS layer as the code does), mouse VIS cells with `layer_ccf`
1 and failed_qc False, both in the four core subclasses; features more than 80% non-missing; a
two-way ANOVA of species and subclass with interaction, species entered first; Benjamini-Hochberg
across features; for features with an interaction at FDR p < 0.05, two-sided Mann-Whitney within
each subclass. O2 readings: (a) ranks, Type II, HC3; (b) raw values, Type I, as the code calls
`anova_lm` (the HC3 fit does not change `anova_lm` without its `robust` option). AP height at the
ramp (`peak_deltav_ramp`) is not computed here and comes from the paper's tables.

In [ ]:
def species_stats(H, M, rank, typ, robust):
    comb = pd.concat([H[(H.collaborator == "AIBS") & (H.failed_fx_long_squares == False) & (H.layer == "1")],
                      M[M.vis & (M.layer_ccf == "1") & (M.failed_qc == False)]], sort=False)
    comb = comb[comb.homology_type.isin(L.SUBCLASSES)].copy()
    data = comb[comb[fcols].notna().sum(axis=1) > 0]
    feats80 = [c for c in fcols if data[c].notna().mean() > 0.8]
    if rank:
        data = data.copy(); data[feats80] = data[feats80].rank()
    recs = []
    for f in feats80:
        fit = smf.ols(f"{f} ~ species * homology_type", data=data).fit(cov_type="HC3")
        tab = anova_lm(fit, typ=typ, robust=("hc3" if robust else None))
        recs.append({"feature": f, "p_species": tab.loc["species", "PR(>F)"],
                     "p_inter": tab.loc["species:homology_type", "PR(>F)"]})
    res = pd.DataFrame(recs).set_index("feature")
    res["p_species_fdr"] = multipletests(res.p_species.fillna(1), method="fdr_bh")[1]
    res["p_inter_fdr"] = multipletests(res.p_inter.fillna(1), method="fdr_bh")[1]
    return comb, res

target = {"sag": "lower", "peak_deltav_ramp": "lower", "rheobase_i": "higher"}
for kind, H, M in [("a", HA, MA), ("b", HB, MB)]:
    for name, rank, typ, robust in [("O2 (a): ranks, Type II, HC3", True, 2, True), ("O2 (b): raw, Type I", False, 1, False)]:
        comb, res = species_stats(H, M, rank, typ, robust)
        print(f"features ({kind}), {name}: n human {int((comb.species == 'human').sum())}, mouse {int((comb.species == 'mouse').sum())}, "
              f"{len(res)} features, species FDR p < 1e-38 for {int((res.p_species_fdr < 1e-38).sum())}")
        for f, direction in target.items():
            med = comb.groupby("species")[f].median()
            sub = {}
            for g in L.SUBCLASSES:
                d = comb[comb.homology_type == g].dropna(subset=[f])
                x, y = d[d.species == "mouse"][f], d[d.species == "human"][f]
                if len(x) and len(y):
                    p = stats.mannwhitneyu(x, y).pvalue
                    ok = (x.median() < y.median()) if direction == "lower" else (x.median() > y.median())
                    sub[g] = f"{p:.2g}{'' if ok else ' (wrong way)'}"
            print(f"   {f}: mouse median {med['mouse']:.3g} vs human {med['human']:.3g} (mouse {direction}: "
                  f"{(med['mouse'] < med['human']) if direction == 'lower' else (med['mouse'] > med['human'])}); "
                  f"species FDR p {res.loc[f, 'p_species_fdr']:.2g}; interaction FDR p {res.loc[f, 'p_inter_fdr']:.2g}; "
                  f"within-subclass MW p {sub}")
print("Cells by species and subclass:"); print(pd.crosstab(comb.homology_type, comb.species).to_string())

**Result: blocked, methods underspecified.** In the stated direction for all three
features (mouse sag 0.048 against 0.136 in human, AP height at the ramp 54 against 76 mV,
rheobase 140 against 30 pA), and for each feature with an interaction at least 3 of the 4
subclasses differ at p < 0.05 the same way (PAX6, with 2 mouse cells, does not). Under O2 reading
(b), raw values with Type I sums of squares as the code calls it, all three species effects have
FDR p below 1e-38 (9.5e-45, 1.4e-43, 2.6e-83). Under reading (a), ranks with Type II and HC3,
AP height has FDR p = 2.6e-38, just above 1e-38, while sag (3.7e-39) and rheobase (1e-78) are
below. The criterion makes the claim methods_underspecified when the readings disagree; it is
reproduced under (b) and partially reproduced under (a). 22 of the paper's mouse cells are not
in 000020, which lowers the power of both.

## fig3g-variability-dimensions

> Human and mouse L1 subclasses have a similar amount of morpho-electric variability but along
> different features; sag and spike shape adaptation vary in human but little in mouse, while ISI
> adaptation and AHP vary in mouse.

As l1_subclass_distinctness.ipynb cells 10 and 37: human, the set of fig2c-subclass-ephys-differences;
mouse, VIS cells of the strict mouse set with a subclass, features more than 40% non-missing;
Kruskal-Wallis per feature, epsilon2 = H/(n-1), Benjamini-Hochberg.

In [ ]:
isi_f = ["isi_adapt_ratio", "adapt_mean", "adapt_hero"]
ahp_f = ["fast_trough_deltav_rheo", "fast_trough_deltav_hero", "trough_slowdeltav_rheo", "trough_slowdeltav_hero", "ahp_delay_ratio_hero"]
for kind, H, M in [("a", HA, MA), ("b", HB, MB)]:
    hP = H[H.in_strict_set & (H.collaborator == "AIBS") & (H.failed_fx_long_squares == False)]
    h40 = [c for c in fcols if hP[hP.homology_type.notna()][c].notna().mean() > 0.4]
    hs = hP[hP.homology_type.isin(L.SUBCLASSES)].copy()
    hs["homology_type"] = pd.Categorical(hs.homology_type, categories=L.SUBCLASSES)
    hres, _ = L.run_kw_dunn(hs, h40, "homology_type", posthoc=None)
    mv = M[M.vis & (M[fcols].notna().sum(axis=1) > 0)]
    m40 = [c for c in fcols if mv[c].notna().mean() > 0.4]
    ms = mv[mv.homology_type.notna()].copy()
    ms["homology_type"] = pd.Categorical(ms.homology_type, categories=L.SUBCLASSES)
    mres, _ = L.run_kw_dunn(ms, m40, "homology_type", posthoc=None)
    both = hres[["epsilon2", "pval_fdr"]].join(mres[["epsilon2", "pval_fdr"]], lsuffix="_human", rsuffix="_mouse", how="outer")
    print(f"features ({kind}): human {len(hs)} cells, {len(h40)} features; mouse {len(ms)} cells {ms.homology_type.value_counts(sort=False).to_dict()}, {len(m40)} features")
    print(both.loc[[f for f in ["sag", "upstroke_adapt_ratio"] + isi_f + ahp_f if f in both.index]].round(4).to_string())
    i = all(both.loc[f, "pval_fdr_human"] < 0.05 and both.loc[f, "epsilon2_mouse"] < both.loc[f, "epsilon2_human"] for f in ["sag", "upstroke_adapt_ratio"])
    def mouse_more(fs):
        return [f for f in fs if f in both.index and both.loc[f, "pval_fdr_mouse"] < 0.05 and both.loc[f, "epsilon2_mouse"] > both.loc[f, "epsilon2_human"]]
    print(f"  (i) sag and upstroke adaptation vary in human and less in mouse: {i}")
    print(f"  (ii) ISI adaptation features varying more in mouse: {mouse_more(isi_f)}; AHP features: {mouse_more(ahp_f)}")
    nh, nm = int((hres.pval_fdr < 0.05).sum()), int((mres.pval_fdr < 0.05).sum())
    eh, em = hres.loc[hres.pval_fdr < 0.05, "epsilon2"].mean(), mres.loc[mres.pval_fdr < 0.05, "epsilon2"].mean()
    print(f"  amount (reported, not part of the outcome): FDR-significant features human {nh}, mouse {nm}; their mean epsilon2 human {eh:.3f}, mouse {em:.3f}")

**Result: reproduced.** Under both feature readings sag and the upstroke adaptation ratio
differ across human subclasses (FDR p < 1e-4) with epsilon2 0.57 and 0.46, against 0.24 and 0.16
in mouse. In mouse the ISI adaptation features (isi_adapt_ratio, adapt_mean, adapt_hero; epsilon2
0.38 to 0.40) and AHP features (fast_trough_deltav_rheo and _hero, trough_slowdeltav_hero;
0.35 to 0.38) differ with FDR p < 1e-4 and larger epsilon2 than in human (0.02 to 0.27). The
amount, reported apart, is 56 FDR-significant features in human and 77 in mouse, with mean
epsilon2 0.17 and 0.21.

## fig4d-mouse-no-rosehip-ephys

> The mouse MC4R subclass has moderate sag but no irregular spiking; irregular spiking occurs only
> in a subset of mouse LAMP5 cells (mainly Lamp5 Ntn1 Npy2r) without other rosehip features.

As l1_rosehip.ipynb cells 4 and 36: mouse cells of the strict set with primary_ephys (VIS,
failed_qc False), sag above 0.5 set missing; Dunn's tests over all subclass pairs; irregular is
`ir_late_mean` above 0.4 (cells 39 and 42). `ir_late_mean` comes from the paper's table (O1).

In [ ]:
from itertools import combinations
for kind, M in [("a", MA), ("b", MB)]:
    m = M[M.vis & (M.failed_qc == False)].copy()
    m.loc[m.sag > 0.5, "sag"] = np.nan
    m = m[m.homology_type.notna()]
    med = m.groupby("homology_type")[["sag", "ir_late_mean"]].median()
    print(f"features ({kind}): {len(m)} mouse cells"); print(med.round(3).to_string())
    pairs = list(combinations(L.SUBCLASSES, 2))
    print("  Dunn FDR p, sag:", {k: f"{v:.2g}" for k, v in L.dunn_pairs(m, "sag", "homology_type", pairs).items()})
    irr = m[m.ir_late_mean > 0.4]
    mc4r = m[m.homology_type == "MC4R"]
    print(f"  MC4R cells with ir_late_mean > 0.4: {int((mc4r.ir_late_mean > 0.4).sum())} of {int(mc4r.ir_late_mean.notna().sum())}")
    print("  cells with ir_late_mean > 0.4 by subclass:", irr.homology_type.value_counts().to_dict(),
          "| by t-type:", irr["t-type"].value_counts().to_dict())
    print(f"  MC4R median sag above LAMP5: {med.loc['MC4R', 'sag'] > med.loc['LAMP5', 'sag']}")
    print("  sag range", round(m.sag.min(), 3), "to", round(m.sag.max(), 3), "| cells set missing for sag > 0.5:",
          int((M[M.vis & (M.failed_qc == False)].sag > 0.5).sum()))

**Result: partially reproduced.** The mouse MC4R subclass has a median sag of 0.141 to
0.147, above LAMP5's 0.041 to 0.042 (Dunn FDR p about 1e-9). Its irregularity ratio has a median of
0.136, and 4 of its 18 cells are above 0.4: a minority (reading (b) holds) but not none (reading
(a) fails). Of the 30 mouse cells above 0.4, 15 are LAMP5, 11 L1 VIP and 4 MC4R, so irregular
cells are not mostly LAMP5, and the largest t-type among them is Lamp5 Plch2 Dock5 (8 cells), not
Lamp5 Ntn1 Npy2r (6). The moderate sag holds; the restriction of irregular firing to LAMP5 cells
does not.

## fig5d-mouse-pax6-doublets

> Mouse PAX6 subclass cells (Lamp5 Krt73) fire doublets at stimulus onset and show a depolarizing
> hump just below rheobase, marking them as alpha7 cells.

As l1_tnfaip.ipynb cells 20 and 23: mouse cells with primary_ephys by subclass; the initial firing
rate at rheobase + 40 pA (computed here under (a)) and the depolarizing sag ratio (`sag_depol`,
from the paper's table, O1).

In [ ]:
for kind, M in [("a", MA), ("b", MB)]:
    m = M[M.vis & (M.failed_qc == False) & M.homology_type.notna()]
    med = m.groupby("homology_type")[["first_isi_inv_hero", "sag_depol"]].median()
    n = m.groupby("homology_type")[["first_isi_inv_hero", "sag_depol"]].count()
    print(f"features ({kind}): medians (n)"); print(med.round(3).astype(str).add(" (").add(n.astype(str)).add(")").to_string())
    krt = m[m["t-type"] == "Lamp5 Krt73"]
    print("  Lamp5 Krt73 cells:"); print(krt[["first_isi_inv_hero", "sag_depol", "n_spikes_hero"]].round(3).to_string())
    others = med.drop(index="PAX6")
    print(f"  PAX6 (Lamp5 Krt73) median above every other subclass: first_isi_inv_hero "
          f"{med.loc['PAX6', 'first_isi_inv_hero'] > others.first_isi_inv_hero.max()}, sag_depol {med.loc['PAX6', 'sag_depol'] > others.sag_depol.max()}")
    for f in ["first_isi_inv_hero", "sag_depol"]:
        print(f"  {f}: KW p {stats.kruskal(*[g[f].dropna() for _, g in m.groupby('homology_type')]).pvalue:.2g}")

**Result: partially reproduced.** Only 3 Lamp5 Krt73 cells are in 000020 (the claims file
expects 4), 2 of them with a value at rheobase + 40 pA. Their depolarizing sag ratio (0.257 to
0.304, median 0.273) is above the medians of every other mouse subclass (0.035 to 0.132). Their
median initial firing rate, 97.7 Hz (58 and 137 Hz), is above LAMP5 (20 Hz) and MC4R (26 Hz) but
below L1 VIP (171 to 177 Hz), so the doublet part of the criterion fails.

## Summary

| Claim | Outcome |
|---|---|
| mouse-cell-link | partially reproduced |
| fig3d-ephys-species | blocked, methods underspecified (reproduced under O2 (b), partially under (a)) |
| fig3g-variability-dimensions | reproduced |
| fig4d-mouse-no-rosehip-ephys | partially reproduced |
| fig5d-mouse-pax6-doublets | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the same feature extraction as the human notebook,
applied to the 250 mouse files of 000020; 22 of the paper's 272 strict mouse L1 cells have no file
there. What was found: the mouse files link to the paper's table by session id and cell name, and
their sweeps that stopped early are also stored with zeros.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min")